In [1]:
import torch
import numpy as np

from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood
from botorch.sampling import SobolQMCNormalSampler
from botorch.acquisition.monte_carlo import qSimpleRegret
from botorch.optim import optimize_acqf


C:\Users\Administrator\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cpu")
dtype = torch.double

# 例子：你自己的 N 组实验配方（每行 10 个溶剂比例，和为 1）
# 这里随便写几行，你换成真实数据即可
X_exp = torch.tensor([
    [0.10, 0.05, 0.20, 0.10, 0.05, 0.10, 0.15, 0.10, 0.10, 0.05],
    [0.12, 0.08, 0.18, 0.10, 0.07, 0.10, 0.13, 0.11, 0.08, 0.03],
    [0.08, 0.05, 0.22, 0.12, 0.04, 0.12, 0.14, 0.13, 0.06, 0.04],
], dtype=dtype, device=device)

In [3]:
# 假设你真实测到了 3 个指标
Y_multi_exp = torch.tensor([
    [0.92, 180.0, 50.0],   # retention, capacity, impedance
    [0.88, 175.0, 45.0],
    [0.95, 178.0, 60.0],
], dtype=dtype, device=device)
Y_norm = Y_multi_exp.clone()
Y_norm[:, 0] = Y_norm[:, 0] / 1.0          # retention 已经是 0~1，可以不动
Y_norm[:, 1] = Y_norm[:, 1] / Y_norm[:, 1].max()  # 初始容量
Y_norm[:, 2] = Y_norm[:, 2] / Y_norm[:, 2].max()  # 阻抗
w_ret = 0.5   # 保持率权重
w_cap = 0.4   # 初始容量权重
w_imp = 0.1   # 阻抗惩罚权重

s_exp = (
    w_ret * Y_norm[:, 0]
  + w_cap * Y_norm[:, 1]
  - w_imp * Y_norm[:, 2]   # 阻抗越大越扣分
).unsqueeze(-1)             # 变成 (N,1)


gp = SingleTaskGP(X_exp, s_exp)
mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
fit_gpytorch_mll(mll)

# 后面批处理 BO 的代码完全不变，只是 y 换成了 s_exp


ExactMarginalLogLikelihood(
  (likelihood): GaussianLikelihood(
    (noise_covar): HomoskedasticNoise(
      (noise_prior): LogNormalPrior()
      (raw_noise_constraint): GreaterThan(1.000E-04)
    )
  )
  (model): SingleTaskGP(
    (likelihood): GaussianLikelihood(
      (noise_covar): HomoskedasticNoise(
        (noise_prior): LogNormalPrior()
        (raw_noise_constraint): GreaterThan(1.000E-04)
      )
    )
    (mean_module): ConstantMean()
    (covar_module): RBFKernel(
      (lengthscale_prior): LogNormalPrior()
      (raw_lengthscale_constraint): GreaterThan(2.500E-02)
    )
    (outcome_transform): Standardize()
  )
)

In [4]:
d = 10  # 维度（如果你上面定义过就不用重复）
bounds = torch.stack([
    torch.zeros(d, device=device, dtype=dtype),
    torch.ones(d, device=device, dtype=dtype),
])
n_new = 5



In [5]:
# ===========================================
# 3. 批处理 BO 搜索策略：qSimpleRegret (Thompson Sampling 风格)
#    一次给出 n_new 个新配方
# ===========================================
sampler = SobolQMCNormalSampler(sample_shape=torch.Size([256]))
acq = qSimpleRegret(
    model=gp,
    sampler=sampler,
)

# 优化采集函数，直接设 q = n_new，就是批处理 BO
X_batch_raw, acq_value = optimize_acqf(
    acq_function=acq,
    bounds=bounds,
    q=n_new,               # 批大小 = 你想要的新配方数量
    num_restarts=10,
    raw_samples=256,
)

# 把候选点投影回 simplex（每行和为 1，且 >=0）
X_batch = X_batch_raw.clamp(min=0)
X_batch = X_batch / X_batch.sum(dim=-1, keepdim=True)

print("\n建议的新配方 X_batch (n_new x d)：")
print(X_batch)
print("每行和（应为1）：", X_batch.sum(dim=-1))


建议的新配方 X_batch (n_new x d)：
tensor([[0.0000, 0.0000, 0.1897, 0.0000, 0.0000, 0.1713, 0.0000, 0.0132, 0.1299,
         0.4959],
        [0.0000, 0.0000, 0.0541, 0.2604, 0.0000, 0.1104, 0.2621, 0.0393, 0.2321,
         0.0416],
        [0.0000, 0.0000, 0.6203, 0.0000, 0.0000, 0.0000, 0.3519, 0.0278, 0.0000,
         0.0000],
        [0.1408, 0.1408, 0.1245, 0.0000, 0.1408, 0.1408, 0.1408, 0.0000, 0.1408,
         0.0306],
        [0.0000, 0.2000, 0.2000, 0.0000, 0.0000, 0.2000, 0.0000, 0.2000, 0.2000,
         0.0000]], dtype=torch.float64)
每行和（应为1）： tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000], dtype=torch.float64)
